In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# User settings
# ============================================================
EVENT_TIME_S = 3.0

# Post-event evaluation window (absolute time)
POST_START_OFFSET_S = 0.0
POST_END_OFFSET_S   = 0.080   # evaluate [EVENT_TIME, EVENT_TIME+POST_END_OFFSET_S]

# Optional pre-event window for de-biasing (set to None to disable)
PRE_BIAS_WINDOW = None  # e.g. (EVENT_TIME_S - 0.05, EVENT_TIME_S - 0.005)

# Base quantities
BASE_POWER_3PH_W = 100e6

# PLL reconstruction parameters (must match controller settings used in the run)
PLL_KP    = 0.25
PLL_KI    = 0.2
PLL_F0_HZ = 50.0

# CSV paths (adjust if needed)
PATH_EMT = "/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx/logs/EMT_simulation/EMT_simulation.csv"
PATH_DP  = "/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx/logs/DP_simulation/DP_simulation.csv"
PATH_SP  = "/home/gti/development/dpsim_local/dpsim/build/dpsim/examples/cxx/logs/SP_simulation/SP_simulation.csv"


# ============================================================
# Helpers
# ============================================================
def _strip_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = df.columns.str.strip()
    return df

def pick_col(df: pd.DataFrame, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def load_csv(path: str) -> pd.DataFrame:
    return _strip_cols(pd.read_csv(path))

def _window_mask(t: np.ndarray, t0: float, t1: float) -> np.ndarray:
    return (t >= t0) & (t <= t1)

def _maybe_debias(t, y, pre_window):
    if pre_window is None:
        return y
    t0, t1 = pre_window
    m = _window_mask(t, t0, t1)
    if np.any(m):
        return y - float(np.mean(y[m]))
    return y

def interpolate_ref(t_ref, y_ref, t_target):
    # assumes t_ref is monotonic
    return np.interp(t_target, t_ref, y_ref)

def rmse_and_peak_abs_err(t, y_model, y_ref_interp, t0, t1):
    m = _window_mask(t, t0, t1)
    if np.sum(m) < 5:
        return np.nan, np.nan
    e = y_model[m] - y_ref_interp[m]
    rmse = float(np.sqrt(np.mean(e**2)))
    peak = float(np.max(np.abs(e)))
    return rmse, peak


# ============================================================
# Signals of interest
#   - Converter 1/2: P,Q in p.u. from vd,id,vq,iq
#   - Converter 1/2: PLL frequency in Hz (same formula as your script)
# ============================================================
def compute_pq_pu_from_dq(df: pd.DataFrame, conv_idx: int):
    vd_col = pick_col(df, [f"vdConverter{conv_idx}", f"vdConverter{conv_idx}.re"])
    vq_col = pick_col(df, [f"vqConverter{conv_idx}", f"vqConverter{conv_idx}.re"])
    id_col = pick_col(df, [f"idConverter{conv_idx}", f"idConverter{conv_idx}.re"])
    iq_col = pick_col(df, [f"iqConverter{conv_idx}", f"iqConverter{conv_idx}.re"])

    if None in (vd_col, vq_col, id_col, iq_col):
        raise KeyError(
            f"Missing dq columns for converter {conv_idx}: "
            f"{vd_col}, {vq_col}, {id_col}, {iq_col}"
        )

    vd = df[vd_col].to_numpy()
    vq = df[vq_col].to_numpy()
    id_ = df[id_col].to_numpy()
    iq_ = df[iq_col].to_numpy()

    p_w   = vd * id_ + vq * iq_
    q_var = vq * id_ - vd * iq_

    return p_w / BASE_POWER_3PH_W, q_var / BASE_POWER_3PH_W

def _find_pll_theta_xi_cols(df: pd.DataFrame, conv_idx: int):
    base = f"pllOutputConverter{conv_idx}"
    c0 = f"{base}_0"
    c1 = f"{base}_1"
    if c0 in df.columns and c1 in df.columns:
        return c0, c1

    cols = sorted([c for c in df.columns if base in c])
    if len(cols) >= 2:
        return cols[0], cols[1]
    return None, None

def compute_pll_frequency_hz(df: pd.DataFrame, conv_idx: int):
    if "time" not in df.columns:
        raise KeyError("Missing 'time' column.")

    t = df["time"].to_numpy()

    vq_col = pick_col(df, [f"vqConverter{conv_idx}", f"vqConverter{conv_idx}.re"])
    if vq_col is None:
        raise KeyError(f"Missing vqConverter{conv_idx} column for PLL frequency.")

    theta_col, xi_col = _find_pll_theta_xi_cols(df, conv_idx)
    if theta_col is None or xi_col is None:
        raise KeyError(
            f"Missing pllOutputConverter{conv_idx}_0/_1 (or equivalent) for PLL frequency."
        )

    vq = df[vq_col].to_numpy()
    xi = df[xi_col].to_numpy()

    omega_nom = 2.0 * np.pi * float(PLL_F0_HZ)
    omega_pll = omega_nom + float(PLL_KP) * vq + float(PLL_KI) * xi
    f_pll = omega_pll / (2.0 * np.pi)
    return t, f_pll


# ============================================================
# Main evaluation (SP as benchmark)
# ============================================================
def evaluate():
    dfE = load_csv(PATH_EMT)
    dfD = load_csv(PATH_DP)
    dfS = load_csv(PATH_SP)

    t0 = EVENT_TIME_S + POST_START_OFFSET_S
    t1 = EVENT_TIME_S + POST_END_OFFSET_S

    rows = []

    def add_metric(domain, signal, t_m, y_m, t_ref, y_ref):
        # optional de-bias both (each by its own pre-window mean)
        y_m2   = _maybe_debias(t_m,   y_m,   PRE_BIAS_WINDOW)
        y_ref2 = _maybe_debias(t_ref, y_ref, PRE_BIAS_WINDOW)

        y_ref_on_m = interpolate_ref(t_ref, y_ref2, t_m)
        rmse, peak = rmse_and_peak_abs_err(t_m, y_m2, y_ref_on_m, t0, t1)

        rows.append({
            "domain": domain,
            "signal": signal,
            "rmse": rmse,
            "peak_abs_err": peak,
            "t_window_s": f"[{t0:.3f}, {t1:.3f}]",
            "bias_removed": PRE_BIAS_WINDOW is not None
        })

    # --- Converter 1 & 2: P,Q (p.u.) ---
    for k in (1, 2):
        # SP reference
        tS = dfS["time"].to_numpy()
        pS, qS = compute_pq_pu_from_dq(dfS, k)

        # EMT vs SP
        tE = dfE["time"].to_numpy()
        pE, qE = compute_pq_pu_from_dq(dfE, k)
        add_metric("EMT", f"Conv{k} P (p.u.)", tE, pE, tS, pS)
        add_metric("EMT", f"Conv{k} Q (p.u.)", tE, qE, tS, qS)

        # DP vs SP
        tD = dfD["time"].to_numpy()
        pD, qD = compute_pq_pu_from_dq(dfD, k)
        add_metric("DP",  f"Conv{k} P (p.u.)", tD, pD, tS, pS)
        add_metric("DP",  f"Conv{k} Q (p.u.)", tD, qD, tS, qS)

    # --- PLL frequency (Hz), Converter 1 & 2 ---
    for k in (1, 2):
        tS_f, fS = compute_pll_frequency_hz(dfS, k)

        tE_f, fE = compute_pll_frequency_hz(dfE, k)
        add_metric("EMT", f"Conv{k} PLL f (Hz)", tE_f, fE, tS_f, fS)

        tD_f, fD = compute_pll_frequency_hz(dfD, k)
        add_metric("DP",  f"Conv{k} PLL f (Hz)", tD_f, fD, tS_f, fS)

    out = pd.DataFrame(rows)

    # Pretty table
    pd.set_option("display.max_rows", 200)
    pd.set_option("display.width", 160)
    pd.set_option("display.float_format", lambda x: f"{x:.6g}" if np.isfinite(x) else "nan")

    print("\n=== Accuracy vs SP (benchmark) ===")
    print(f"Event time: {EVENT_TIME_S:.6g} s")
    print(f"Post window: [{t0:.6g}, {t1:.6g}] s")
    print(f"Pre-window de-bias: {PRE_BIAS_WINDOW}\n")
    print(out[["domain", "signal", "rmse", "peak_abs_err", "t_window_s", "bias_removed"]].to_string(index=False))

    return out


if __name__ == "__main__":
    evaluate()
